<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

#<center>Manejo de Archivos</center>

### **¿Por qué trabajar con archivos?**

Los programas que vimos hasta ahora trabajan con datos **en memoria**: cuando el programa termina, los datos desaparecen. Los archivos nos permiten **persistir información**: guardarla en el disco para usarla después, compartirla o procesarla con otro programa.

Python distingue dos grandes tipos de archivos:

| Tipo | Descripción | Ejemplos |
|---|---|---|
| **Texto** | Contenido legible como caracteres | `.txt`, `.csv`, `.json`, `.py` |
| **Binario** | Contenido en bytes, no legible directamente | `.png`, `.jpg`, `.pdf`, `.exe` |

En esta notebook veremos:
- Abrir y cerrar archivos con `open()`
- El context manager `with open()`
- Leer y escribir archivos de texto
- Archivos CSV con el módulo `csv`
- Archivos binarios: imágenes
- Manejo de errores aplicado a archivos

---
### **`open()` y los modos de apertura**

La función `open(ruta, modo)` abre un archivo y devuelve un **objeto de archivo** a través del cual podemos leer o escribir.

| Modo | Descripción | ¿Crea el archivo? | ¿Borra el contenido? |
|---|---|---|---|
| `'r'` | Lectura (por defecto) | No (error si no existe) | No |
| `'w'` | Escritura | Sí | Sí (sobreescribe) |
| `'a'` | Agregar al final | Sí | No |
| `'r+'` | Lectura y escritura | No (error si no existe) | No |
| `'rb'` | Lectura binaria | No | No |
| `'wb'` | Escritura binaria | Sí | Sí |

> ⚠️ Si abrimos un archivo con `open()` directamente (sin `with`), debemos cerrarlo manualmente con `.close()`. Si no lo hacemos, podemos perder datos o bloquear el archivo.

In [ ]:
# Forma manual (NO recomendada): hay que cerrar el archivo explícitamente
archivo = open("ejemplo.txt", "w")
archivo.write("Hola mundo\n")
archivo.close()   # ← si esto no se ejecuta (por un error), el archivo queda bloqueado

print("Archivo creado")

---
### **Context manager: `with open()`**

La forma **correcta y recomendada** de trabajar con archivos en Python es usando `with open()`. Este patrón garantiza que el archivo se **cierre automáticamente** al salir del bloque, incluso si ocurre una excepción.

```python
with open("archivo.txt", "modo") as f:
    # operaciones sobre f
# acá el archivo ya está cerrado automáticamente
```

> Es exactamente el problema que resuelve `finally` que vimos en manejo de errores: el `with` garantiza que el cierre ocurra siempre.

In [ ]:
# Forma correcta: with open()
with open("ejemplo.txt", "w") as f:
    f.write("Hola mundo\n")
    f.write("Segunda línea\n")
# Al salir del with, el archivo se cierra automáticamente

print("Archivo escrito y cerrado")
print("¿Está cerrado?", f.closed)   # True

---
### **Escribir archivos de texto**

In [ ]:
# Escribir líneas con write()
# Nota: write() no agrega salto de línea automático, hay que incluir \n
alumnos: list[str] = ["Ana García", "Bruno López", "Carla Pérez", "Diego Martín"]

with open("alumnos.txt", "w", encoding="utf-8") as f:
    f.write("=== LISTADO DE ALUMNOS ===\n")
    for alumno in alumnos:
        f.write(alumno + "\n")

print("Archivo alumnos.txt creado")

In [ ]:
# writelines(): escribir una lista de strings de una vez
# (tampoco agrega \n automáticamente)
lineas: list[str] = [
    "Línea 1\n",
    "Línea 2\n",
    "Línea 3\n",
]

with open("lineas.txt", "w", encoding="utf-8") as f:
    f.writelines(lineas)

print("Archivo lineas.txt creado")

#### Modo `'a'`: agregar contenido sin borrar lo existente

In [ ]:
# Primero creamos el archivo
with open("log.txt", "w", encoding="utf-8") as f:
    f.write("[INFO] Sistema iniciado\n")

# Luego agregamos más líneas sin borrar las anteriores
nuevas_entradas: list[str] = [
    "[INFO] Usuario conectado\n",
    "[WARNING] Intento fallido de login\n",
    "[INFO] Operación completada\n",
]

for entrada in nuevas_entradas:
    with open("log.txt", "a", encoding="utf-8") as f:
        f.write(entrada)

print("Entradas agregadas al log")

---
### **Leer archivos de texto**

Hay tres formas principales de leer el contenido de un archivo de texto:

| Método | Devuelve | Cuándo usarlo |
|---|---|---|
| `read()` | Un string con todo el contenido | Archivos chicos |
| `readline()` | Una línea por vez | Cuando queremos procesar de a una |
| `readlines()` | Lista de strings (una por línea) | Cuando necesitamos todas las líneas |
| `for línea in f` | Itera línea por línea | Archivos grandes (eficiente en memoria) |

In [ ]:
# read(): leer todo el contenido de una vez
with open("alumnos.txt", "r", encoding="utf-8") as f:
    contenido: str = f.read()

print("Contenido completo:")
print(contenido)
print("Tipo:", type(contenido))

In [ ]:
# readlines(): leer todas las líneas como lista
with open("alumnos.txt", "r", encoding="utf-8") as f:
    lineas: list[str] = f.readlines()

print("Lista de líneas:")
print(lineas)
print("\nCantidad de líneas:", len(lineas))

In [ ]:
# Iterar línea por línea con for (más eficiente para archivos grandes)
print("Alumnos en el archivo:")
with open("alumnos.txt", "r", encoding="utf-8") as f:
    for linea in f:
        linea_limpia: str = linea.strip()   # strip() elimina \n y espacios
        if linea_limpia and not linea_limpia.startswith("==="):
            print(" -", linea_limpia)

In [ ]:
# readline(): leer una línea a la vez (útil para archivos con encabezado)
with open("alumnos.txt", "r", encoding="utf-8") as f:
    encabezado: str = f.readline()   # lee solo la primera línea
    print("Encabezado:", encabezado.strip())

    print("Resto del archivo:")
    for linea in f:                  # continúa desde donde quedó
        print(" ", linea.strip())

#### Ejemplo aplicado: leer el log y contar tipos de mensajes

In [ ]:
contadores: dict[str, int] = {"INFO": 0, "WARNING": 0, "ERROR": 0}

with open("log.txt", "r", encoding="utf-8") as f:
    for linea in f:
        for tipo in contadores:
            if "[" + tipo + "]" in linea:
                contadores[tipo] += 1

print("Resumen del log:")
for tipo, cantidad in contadores.items():
    print(f"  {tipo}: {cantidad}")

---
### **Manejo de errores con archivos**

Las operaciones con archivos son propensas a errores: el archivo puede no existir, no tener permisos de lectura, o estar corrupto. Siempre conviene protegerlas con `try/except`.

In [ ]:
def leer_archivo(ruta: str) -> str:
    """Lee y devuelve el contenido de un archivo de texto.

    Args:
        ruta: ruta al archivo

    Returns:
        Contenido del archivo como string.

    Raises:
        FileNotFoundError: si el archivo no existe
        PermissionError: si no hay permisos de lectura
    """
    try:
        with open(ruta, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"Error: el archivo '{ruta}' no existe")
        return ""
    except PermissionError:
        print(f"Error: no tenés permisos para leer '{ruta}'")
        return ""
    except UnicodeDecodeError:
        print(f"Error: el archivo '{ruta}' no es un archivo de texto válido")
        return ""


# Archivo existente
contenido: str = leer_archivo("alumnos.txt")
print("Leído:", len(contenido), "caracteres")

# Archivo inexistente
leer_archivo("no_existe.txt")

---
### **Archivos CSV con el módulo `csv`**

Los archivos **CSV** (Comma-Separated Values) son archivos de texto donde cada línea es un registro y los campos se separan por comas (o punto y coma).

Python incluye el módulo `csv` en su librería estándar, que maneja correctamente casos especiales como campos con comas, comillas o saltos de línea dentro de un campo.

> Aunque en Analítica de Negocios usaremos Pandas para trabajar con CSVs complejos, el módulo `csv` es útil cuando no queremos importar librerías externas o trabajamos con archivos simples.

In [ ]:
import csv

# Primero creamos un CSV de ejemplo
datos_productos: list[list] = [
    ["producto",     "categoria", "precio", "stock"],  # encabezado
    ["Laptop",       "Tech",      1500.0,   10],
    ["Mouse",        "Tech",      25.0,     150],
    ["Monitor",      "Tech",      800.0,    25],
    ["Teclado",      "Tech",      120.0,    80],
    ["Auriculares",  "Audio",     200.0,    40],
    ["Webcam",       "Tech",      90.0,     60],
]

with open("productos.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(datos_productos)

print("Archivo productos.csv creado")

#### `csv.reader`: leer fila por fila

In [ ]:
with open("productos.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)

    encabezado: list[str] = next(reader)   # leer la primera fila (encabezado)
    print("Columnas:", encabezado)
    print()

    for fila in reader:
        print(fila)

#### `csv.DictReader`: leer como diccionarios

Es la forma más cómoda: cada fila se convierte en un diccionario donde las claves son los nombres de las columnas del encabezado.

In [ ]:
with open("productos.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for fila in reader:
        print(fila["producto"], "→ $", fila["precio"], "| Stock:", fila["stock"])

#### Procesar datos mientras se lee el CSV

In [ ]:
# Calcular el valor total del inventario
total_inventario: float = 0.0
productos_sin_stock: list[str] = []

with open("productos.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for fila in reader:
        precio: float = float(fila["precio"])
        stock: int    = int(fila["stock"])
        valor: float  = precio * stock
        total_inventario += valor

        if stock == 0:
            productos_sin_stock.append(fila["producto"])

print(f"Valor total del inventario: ${total_inventario:,.2f}")
if productos_sin_stock:
    print("Sin stock:", productos_sin_stock)
else:
    print("Todos los productos tienen stock")

#### `csv.writer`: escribir filas

In [ ]:
# Escribir resultados procesados en un nuevo CSV
resultados: list[dict] = []

with open("productos.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for fila in reader:
        precio: float  = float(fila["precio"])
        stock: int     = int(fila["stock"])
        resultados.append({
            "producto":       fila["producto"],
            "precio":         precio,
            "precio_con_iva": round(precio * 1.21, 2),
            "stock":          stock,
            "valor_total":    round(precio * stock, 2),
        })

# Escribir con DictWriter
campos: list[str] = ["producto", "precio", "precio_con_iva", "stock", "valor_total"]

with open("productos_procesados.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=campos)
    writer.writeheader()        # escribe la fila de encabezado
    writer.writerows(resultados)

print("Archivo productos_procesados.csv creado")

# Verificar leyendo el resultado
with open("productos_procesados.csv", "r", encoding="utf-8") as f:
    print(f.read())

#### Ejemplo completo: pipeline de CSV con manejo de errores

In [ ]:
def procesar_csv(ruta_entrada: str, ruta_salida: str) -> int:
    """Lee un CSV de productos, agrega columnas calculadas y guarda el resultado.

    Returns:
        Cantidad de filas procesadas, o -1 si hubo un error.
    """
    try:
        filas_procesadas: int = 0
        resultados: list[dict] = []

        with open(ruta_entrada, "r", encoding="utf-8") as f_entrada:
            reader = csv.DictReader(f_entrada)

            for fila in reader:
                try:
                    precio: float = float(fila["precio"])
                    stock: int    = int(fila["stock"])
                except (ValueError, KeyError) as e:
                    print(f"  Fila inválida ignorada: {dict(fila)} → {e}")
                    continue

                resultados.append({
                    "producto":   fila["producto"],
                    "precio":     precio,
                    "iva":        round(precio * 0.21, 2),
                    "precio_final": round(precio * 1.21, 2),
                    "stock":      stock,
                })
                filas_procesadas += 1

        campos: list[str] = ["producto", "precio", "iva", "precio_final", "stock"]
        with open(ruta_salida, "w", newline="", encoding="utf-8") as f_salida:
            writer = csv.DictWriter(f_salida, fieldnames=campos)
            writer.writeheader()
            writer.writerows(resultados)

        return filas_procesadas

    except FileNotFoundError:
        print(f"Error: no se encontró el archivo '{ruta_entrada}'")
        return -1


n: int = procesar_csv("productos.csv", "productos_con_iva.csv")
if n >= 0:
    print(f"Procesadas {n} filas correctamente")

# Archivo inexistente
procesar_csv("no_existe.csv", "salida.csv")

---
### **Archivos binarios**

Un **archivo binario** almacena datos como secuencias de bytes, no como texto legible. Las imágenes, videos, PDFs y ejecutables son archivos binarios.

Para abrirlos usamos el modo `'rb'` (read binary) o `'wb'` (write binary). La diferencia fundamental con los archivos de texto es que trabajamos con **objetos `bytes`** en lugar de strings.

| | Archivo de texto | Archivo binario |
|---|---|---|
| Modo de apertura | `'r'`, `'w'`, `'a'` | `'rb'`, `'wb'` |
| Tipo de dato | `str` | `bytes` |
| Legible directamente | Sí | No |
| Ejemplos | `.txt`, `.csv` | `.png`, `.jpg`, `.pdf` |

#### Crear una imagen PNG mínima para los ejemplos

#### Copiar un archivo binario byte a byte

In [ ]:
def copiar_archivo_binario(ruta_origen: str, ruta_destino: str) -> bool:
    """Copia un archivo binario de origen a destino.

    Returns:
        True si la copia fue exitosa.
    """
    try:
        with open(ruta_origen, "rb") as origen:
            with open(ruta_destino, "wb") as destino:
                bloque: bytes = origen.read(4096)   # leer el primer bloque de 4KB
                while bloque:                       # mientras haya contenido
                    destino.write(bloque)
                    bloque = origen.read(4096)      # leer el siguiente bloque
        return True
    except FileNotFoundError as e:
        print("Error:", e)
        return False


exito: bool = copiar_archivo_binario("imagen_ejemplo.png", "imagen_copia.png")
print("Copia exitosa:", exito)

# Verificar que la copia es idéntica
with open("imagen_ejemplo.png", "rb") as f1:
    with open("imagen_copia.png",  "rb") as f2:
        son_iguales: bool = f1.read() == f2.read()
print("Los archivos son idénticos:", son_iguales)

---
### **Resumen**

#### Modos de apertura

| Modo | Operación | Crea archivo | Borra contenido |
|---|---|---|---|
| `'r'` | Lectura de texto | No | No |
| `'w'` | Escritura de texto | Sí | Sí |
| `'a'` | Agregar al final | Sí | No |
| `'rb'` | Lectura binaria | No | No |
| `'wb'` | Escritura binaria | Sí | Sí |

#### Métodos de lectura

| Método | Devuelve |
|---|---|
| `f.read()` | Todo el contenido como string |
| `f.readline()` | Una línea |
| `f.readlines()` | Lista de líneas |
| `for línea in f` | Itera línea por línea (eficiente) |

#### Módulo `csv`

| Clase | Uso |
|---|---|
| `csv.reader(f)` | Leer fila como lista |
| `csv.writer(f)` | Escribir fila como lista |
| `csv.DictReader(f)` | Leer fila como diccionario |
| `csv.DictWriter(f, fieldnames)` | Escribir fila como diccionario |

#### Reglas de oro

- Siempre usá `with open()` → cierre automático garantizado
- Siempre especificá `encoding='utf-8'` en archivos de texto
- Siempre usá `newline=''` al abrir CSVs
- Siempre protegé las operaciones con `try/except FileNotFoundError`
- Para imágenes y binarios, usá modo `'rb'` / `'wb'`